### Transform Orders Data - String to JSON
##### 1. Pre-process the JSON string to fix the data quality issues
##### 2.Transform JSONstring to Json Object
##### 3.Wrie transformed data to the silver schema

In [0]:
df = spark.read.table('gizmobox_catalog_noori.bronze.orders')
display(df)

In [0]:
from pyspark.sql.functions import regexp_replace

df_formatted = df.select(
    'value',
    regexp_replace('value', '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "$1"').alias('fixed_value')
)
display(df_formatted)

In [0]:
from pyspark.sql.functions import schema_of_json
df_formatted_json = df_formatted.select(
  schema_of_json(df_formatted.fixed_value).alias('schema'),
  'fixed_value'
)
display(df_formatted_json)




In [0]:
from pyspark.sql.functions import from_json

df_json = df_formatted_json.select(from_json(df_formatted_json.fixed_value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>').alias('json_value'),
df_formatted_json.fixed_value.alias('fixed_value'))
display(df_json)




In [0]:
df_json.writeTo('gizmobox_catalog_noori.silver.orders').createOrReplace()

In [0]:
spark.read.table('gizmobox_catalog_noori.silver.orders').display()
